In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
# ✅ Cell 1: Import libraries
!pip install --upgrade scikit-learn==1.4.2 imbalanced-learn==0.11.0 --quiet

import numpy as np
import pandas as pd
import os
import time
import gc
import joblib
import warnings
from collections import Counter

from sklearn.model_selection import KFold, train_test_split
from sklearn.ensemble import StackingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Suppress convergence warnings from Logistic Regression
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message=".*lbfgs failed to converge.*")

print("✅ All libraries imported")

In [ ]:
# ✅ Load GA-selected dataset
df_ga = pd.read_csv('/kaggle/input/relief/relieff_selected_25.csv')
X_ga = df_ga.drop(columns=['Label'])
y_ga = df_ga['Label']

# ✅ Sample only 10% of data for quick testing
#X_sample, _, y_sample, _ = train_test_split(X_ga, y_ga, stratify=y_ga, test_size=0.9, random_state=42)
# NEW (Full dataset):
X_sample, y_sample = X_ga, y_ga
print(f"✅ Sample dataset loaded | Shape: {X_sample.shape}")


In [ ]:
# ✅ Define base classifiers
base_classifiers = [
    ('lr', LogisticRegression(max_iter=2000, solver='lbfgs')),
    ('gnb', GaussianNB()),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('et', ExtraTreesClassifier(n_estimators=50, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=50, use_label_encoder=False, eval_metric='mlogloss', verbosity=0))
]

# ✅ Meta classifier
meta_clf = DecisionTreeClassifier(random_state=42)
print("✅ Base and meta classifiers defined")


In [5]:
# ✅ Use 3-fold instead of 5-fold
kf = KFold(n_splits=3, shuffle=True, random_state=42)
all_true = []
all_pred = []

out_dir = 'fold_outputs_dt_meta/'
os.makedirs(out_dir, exist_ok=True)

fold = 1
for train_idx, test_idx in kf.split(X_sample):
    print(f"\n📂 Fold {fold} starting...")
    start_fold_time = time.time()

    X_train, X_test = X_sample.iloc[train_idx], X_sample.iloc[test_idx]
    y_train, y_test = y_sample.iloc[train_idx], y_sample.iloc[test_idx]
    print("🔍 Train/Test split done")

    # ✅ Apply SMOTE
    print(f"📊 Label distribution before SMOTE: {Counter(y_train)}")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print(f"✅ SMOTE done | After: {Counter(y_train_resampled)}")

    # ✅ Train StackingClassifier
    model_start = time.time()
    stacking_model = StackingClassifier(
        estimators=base_classifiers,
        final_estimator=clone(meta_clf),
        cv=3,                    # Inner CV for meta model
        passthrough=True,
        n_jobs=1                 # Reduced for memory safety
    )
    print("🚀 Training StackingClassifier...")
    stacking_model.fit(X_train_resampled, y_train_resampled)
    print(f"✅ Model trained in {time.time() - model_start:.2f} seconds")

    # ✅ Predict and save results
    y_pred = stacking_model.predict(X_test)
    all_true.extend(y_test)
    all_pred.extend(y_pred)

    joblib.dump((y_test, y_pred), os.path.join(out_dir, f'fold{fold}_results_relief.pkl'))
    print(f"💾 Saved predictions for Fold {fold}")

    print(f"📉 Classification Report for Fold {fold}:")
    print(classification_report(y_test, y_pred))

    print(f"⏱️ Fold {fold} completed in {time.time() - start_fold_time:.2f} seconds")
    fold += 1

    # ✅ Free memory
    del X_train, X_test, y_train, y_test, X_train_resampled, y_train_resampled, stacking_model, y_pred
    gc.collect()


📂 Fold 1 starting...
🔍 Train/Test split done
📊 Label distribution before SMOTE: Counter({0: 128468, 1: 115312, 8: 56154, 4: 29611, 2: 18302, 6: 11203, 3: 8512, 5: 3453, 9: 1429, 7: 486})
✅ SMOTE done | After: Counter({0: 128468, 1: 128468, 2: 128468, 3: 128468, 4: 128468, 5: 128468, 6: 128468, 7: 128468, 8: 128468, 9: 128468})
🚀 Training StackingClassifier...
✅ Model trained in 4694.97 seconds
💾 Saved predictions for Fold 1
📉 Classification Report for Fold 1:
              precision    recall  f1-score   support

           0       0.97      0.64      0.77     64264
           1       0.95      0.97      0.96     57724
           2       0.92      0.88      0.90      9321
           3       0.64      0.56      0.60      4261
           4       0.48      1.00      0.65     14765
           5       0.25      0.56      0.34      1677
           6       0.18      0.32      0.23      5629
           7       0.13      0.70      0.22       254
           8       0.81      0.68      0.74     

In [6]:

# ✅ Save final outputs
joblib.dump((all_true, all_pred), os.path.join(out_dir, f'all_folds_results_relief.pkl'))
print("✅ All folds completed and saved")


✅ All folds completed and saved
